# M3 · Ola 1 — Corpus + RAG ingenuo + eval set (Isabella)

**SI4006 · Universidad EAFIT · Triage dermatológico**

Este notebook corre de principio a fin la parte de la Ola 1 y deja en el repo los archivos que necesita la Ola 2:

1. Verifica el eval set nuevo (citas textuales contra el corpus y novedad frente a M1/M2).
2. Compara dos configuraciones de chunking y elige una con una regla declarada de antemano.
3. Muestra el RAG ingenuo contra el modelo sin RAG en dos preguntas (como en el Lab B de S07).
4. Corre el **harness de M2** (D1, D2 y D3) sobre **BETO+LoRA** y sobre el **RAG ingenuo**, con el mismo eval set.
5. Genera `docs/M3_consultas_fallidas.md` y los scorecards, y los empaqueta en un zip para subirlos al repo.

**Entorno:** Colab con GPU **T4** (Entorno de ejecución → Cambiar tipo de entorno → T4 GPU). En la T4 tarda unos 20–30 minutos. Las decisiones de diseño están explicadas en `docs/M3_rag_ingenuo.md`.

## 0 · Setup

In [ ]:
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO HAY GPU: activa la T4 antes de seguir')

In [ ]:
# Rama del repo donde están los archivos de M3 (cámbiala si se subieron a otra)
REPO_URL = 'https://github.com/Isa-Idarraga/triage-dermatologico-ia.git'
BRANCH = 'isabella-m3'

import os
os.chdir('/content')
!rm -rf triage-dermatologico-ia
!git clone -q -b {BRANCH} {REPO_URL}
os.chdir('/content/triage-dermatologico-ia')
!git log --oneline -1


In [ ]:
# Lo que Colab no trae. Los pines de opentelemetry son los mismos del notebook de S10.
%pip install -q sentence-transformers chromadb peft "opentelemetry-api==1.42.1" "opentelemetry-sdk==1.42.1"
# Colab trae una versión vieja de torchao que hace fallar el import de peft (mismo arreglo del notebook del juez de M2).
!pip uninstall -y -q torchao


In [ ]:
import sys, json
sys.path.insert(0, 'scripts')
import rag_ingenuo as R
print('Generador :', R.GEN_MODEL, R.GEN_REVISION[:8])
print('Embeddings:', R.EMB_MODEL, R.EMB_REVISION[:8])
print('K =', R.K, '| configs de chunking:', list(R.CHUNKING_CONFIGS))

## 1 · Eval set nuevo: verificación

`construir_eval_set_m3.py` se niega a escribir el eval set si alguna cita de evidencia no aparece **textual** en el corpus. También mide qué tanto se parece cada caso a los textos de M1 y M2 (para mostrar que no está reciclado).

In [ ]:
!python scripts/construir_eval_set_m3.py

## 2 · Chunking: dos configuraciones, elegidas midiendo

Mismo modelo de embeddings y mismas consultas; solo cambia cómo se parte el corpus. La métrica es hit@3 / MRR a nivel documento (¿llegó al top-3 algún documento relevante?). La regla de elección está en el código **desde antes** de correr esto.

In [ ]:
info_chunking = R.comparar_chunking()

## 3 · A ojo: sin RAG vs con RAG (Lab B de S07)

La misma pregunta al generador solo (de memoria) y al RAG ingenuo (busca, lee y cita). La segunda pregunta es adversarial: la respuesta **no** está en el corpus.

In [ ]:
preguntas = [
    'Tengo el labio de abajo reseco y pelado hace meses, me echo vaselina y cacao y no se me quita.',
    'Tengo manchas blancas en las manos que se están extendiendo, no pican. ¿Es vitiligo? ¿Qué crema me echo?',
]
for p in preguntas:
    print('=' * 90); print('CONSULTA:', p)
    print('\n--- SIN RAG (de memoria) ---')
    print(R.generar('Eres un asistente de triage dermatológico. Responde breve si el caso es urgente o no urgente.', p))
    r = R.sistema_rag_ingenuo(p)
    print('\n--- CON RAG ---')
    for cid, s in zip(r['chunk_ids'], r['scores']):
        print(f'  [sim {s}] {cid}')
    print(r['respuesta'])
    print('-> etiqueta para el harness:', r['etiqueta_predicha'], '| válvula:', r['valvula'])

## 4 · Harness de M2 sobre BETO+LoRA y sobre el RAG ingenuo

Hace todo lo de la Ola 1 en un solo llamado: baseline de BETO sobre el eval set **nuevo** (y guarda en cada caso si M1 falla), RAG ingenuo con las **dos** configuraciones de chunking, resumen, versiones y `docs/M3_consultas_fallidas.md`. Carga tres modelos en la T4: BETO+LoRA, el juez de M2 (Qwen2.5-3B) y el generador (Qwen2.5-1.5B).

In [ ]:
resumen = R.evaluar(usar_juez=True, ambos_chunking=True)

In [ ]:
import pandas as pd
filas = {'BETO+LoRA (M1)': resumen['beto_m1']}
filas.update({f'RAG ingenuo · {k}': v for k, v in resumen['rag_ingenuo'].items()})
cols = ['d1_aciertos', 'd1_accuracy', 'd2_juez_promedio_gold', 'd3_recall_urgente', 'd3_recall_urgente_estricto',
        'retrieval_hit@3', 'evidencia_en_contexto_prom', 'cita_doc_relevante', 'valvulas_activadas', 'no_aplica_con_valvula_o_no_aplica', 'formato_invalido',
        'latencia_s_promedio', 'latencia_s_p95']
pd.DataFrame(filas).T.reindex(columns=cols)

In [ ]:
# ¿Qué casos del eval set nuevo falla el sistema actual (BETO+LoRA)?
ev = json.load(open('eval/eval_set_m3.json', encoding='utf-8'))
fallas = [e for e in ev if e['falla_m1']]
con_etiqueta = [e for e in ev if e['falla_m1'] is not None]
print(f'BETO+LoRA falla {len(fallas)} de {len(con_etiqueta)} casos con etiqueta')
pd.DataFrame([{k: e[k] for k in ('id', 'esperado', 'prediccion_m1', 'confianza_m1', 'dificultad')} for e in fallas])

In [ ]:
from IPython.display import Markdown, display
display(Markdown(open('docs/M3_consultas_fallidas.md', encoding='utf-8').read()))

## 5 · Empaquetar los resultados para el repo

Descarga un zip con los archivos que generó la corrida. Esos archivos van al repo (rama de M3) tal cual: son la entrada de María Alejandra.

In [ ]:
import glob, zipfile
archivos = (['eval/eval_set_m3.json', 'eval/scorecard_rag_ingenuo.csv', 'eval/scorecard_beto_eval_m3.csv',
             'results/chunking_m3.json', 'results/m3_ola1_resumen.json', 'results/rag_ingenuo_config.json',
             'docs/M3_consultas_fallidas.md'] + glob.glob('eval/scorecard_rag_ingenuo_*.csv'))
with zipfile.ZipFile('resultados_m3_ola1.zip', 'w') as z:
    for a in archivos:
        z.write(a)
print('\n'.join(archivos))
try:
    from google.colab import files
    files.download('resultados_m3_ola1.zip')
except ImportError:
    print('(fuera de Colab: el zip quedó en', os.path.abspath('resultados_m3_ola1.zip'), ')')